In [1]:
import re
import time
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import requests

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 30

def load_tokens_from_env_file(env_path: Path) -> List[Tuple[str, str]]:
    """
    Returns a list of (var_name, token_value) for lines like:
    GITHUB_TOKEN_1=...
    """
    pairs: List[Tuple[str, str]] = []
    text = env_path.read_text(encoding="utf-8", errors="ignore")
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            pairs.append((k, v))
    return pairs

def mask_token(tok: str) -> str:
    if len(tok) <= 10:
        return "*" * len(tok)
    return tok[:4] + "*" * (len(tok) - 8) + tok[-4:]

def gh_get_json(token: str, url: str) -> Tuple[int, Dict, Dict]:
    """
    Tries Authorization: Bearer first; if unauthorized, retries with Authorization: token.
    Returns (status_code, json_body_or_empty, headers).
    """
    session = requests.Session()
    session.headers.update({
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "token-check/1.0",
    })

    def do(auth_value: str):
        session.headers["Authorization"] = auth_value
        resp = session.get(url, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
        try:
            data = resp.json() if resp.content else {}
        except Exception:
            data = {}
        return resp.status_code, data, dict(resp.headers)

    status, data, headers = do(f"Bearer {token}")
    if status in (401, 403) and isinstance(data, dict) and "message" in data:
        # Some tokens (esp. classic PATs) are commonly used with "token"
        status2, data2, headers2 = do(f"token {token}")
        return status2, data2, headers2
    return status, data, headers

def check_one(var_name: str, token: str) -> Dict:
    # /user is a simple auth-check endpoint (works for PATs and fine-grained PATs if allowed)
    status, data, headers = gh_get_json(token, "https://api.github.com/user")

    remaining = headers.get("X-RateLimit-Remaining")
    reset = headers.get("X-RateLimit-Reset")

    result = {
        "var": var_name,
        "token_masked": mask_token(token),
        "http_status": status,
        "valid": status == 200,
        "login": data.get("login", "") if isinstance(data, dict) else "",
        "rate_remaining": remaining if remaining is not None else "",
        "rate_reset_epoch": reset if reset is not None else "",
        "message": data.get("message", "") if isinstance(data, dict) else "",
    }
    return result

def main():
    if not TOKENS_ENV_PATH.exists():
        raise FileNotFoundError(f"Not found: {TOKENS_ENV_PATH}")

    pairs = load_tokens_from_env_file(TOKENS_ENV_PATH)
    if not pairs:
        print("No GITHUB_TOKEN_* entries found.")
        return

    print(f"Found {len(pairs)} tokens in {TOKENS_ENV_PATH}")
    print("-" * 80)

    ok = 0
    for var, tok in pairs:
        try:
            r = check_one(var, tok)
        except requests.exceptions.RequestException as e:
            print(f"{var}: ERROR network/timeout: {e}")
            continue

        if r["valid"]:
            ok += 1
            print(f"{var}: VALID  login={r['login']}  remaining={r['rate_remaining']}  reset={r['rate_reset_epoch']}  token={r['token_masked']}")
        else:
            print(f"{var}: INVALID status={r['http_status']}  msg={r['message']}  token={r['token_masked']}")

        # small delay to be polite / avoid bursts
        time.sleep(0.2)

    print("-" * 80)
    print(f"Valid: {ok}/{len(pairs)}")

if __name__ == "__main__":
    main()


Found 7 tokens in C:\GitHub\Android-Mobile-Apps\All_Tokens.env
--------------------------------------------------------------------------------
GITHUB_TOKEN_1: VALID  login=behnamparsa  remaining=4996  reset=1766197295  token=ghp_********************************L09I
GITHUB_TOKEN_2: VALID  login=Bparsazad  remaining=4999  reset=1766197769  token=ghp_********************************PG6H
GITHUB_TOKEN_3: VALID  login=BehnamParsa4  remaining=4999  reset=1766197769  token=ghp_********************************yQbc
GITHUB_TOKEN_4: INVALID status=403  msg=API rate limit exceeded for user ID 218371970. If you reach out to GitHub Support for help, please include the request ID DE8A:2B20AE:9C3C81:2AD7366:6945FBFA and timestamp 2025-12-20 01:29:30 UTC. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service)  token=ghp_********************************Bg4x
GITHUB_TOKEN_5: INVALID sta